In [1]:
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix , classification_report
from sklearn.model_selection import train_test_split
from tensorflow import keras
import numpy as np

### Data Loading and Preprocessing 

In [2]:
df = pd.read_csv('Bank Customer Churn Prediction.csv', dtype={'age': 'int8', 'churn': 'int8'})

# drop irrelevant collum
df.drop('customer_id', axis=1, inplace=True)

df1 = df.copy()
df1['gender'] = df1['gender'].replace({'Female': 1, 'Male': 0})

# One-hot encoding
df2 = pd.get_dummies(data=df1, columns=['country'], dtype='int')

# Re-scale numerical features
scaler = MinMaxScaler()
cols_to_scal = ['credit_score', 'age', 'balance', 'estimated_salary']
df2[cols_to_scal] = scaler.fit_transform(df2[cols_to_scal])


In [3]:
df2.head(4)

,credit_score,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn,country_France,country_Germany,country_Spain
0,0.538,1,0.324324,2,0.000000,1,1,1,0.506735,1,1,0,0
1,0.516,1,0.310811,1,0.334031,1,0,1,0.562709,0,0,0,1
2,0.304,1,0.324324,8,0.636357,3,1,0,0.569654,1,1,0,0
3,0.698,1,0.283784,1,0.000000,2,0,0,0.469120,0,1,0,0


In [4]:
# Spliting the data
y = df2.churn
X = df2.drop('churn', axis=1)
Xtr, Xts, ytr, yts = train_test_split(X, y, test_size=0.2, random_state=2)

## 1. Baseline with Imbalanced Data

In [5]:
# Creat a training function
def ANN(X_train, X_test, y_train, y_test):

    model = keras.Sequential([
        keras.layers.Dense(8, input_shape=(12,), activation='relu'),
        keras.layers.Dense(4, activation='relu'),
        keras.layers.Dense(2, activation='relu'),
        keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=16)

    y_preds = model.predict(X_test)
    y_predicted = np.round(y_preds)

    print("Classification Report \n", classification_report(y_test, y_predicted))

    return y_predicted


In [6]:
# Train ANN on Imbalanced data
y_predict = ANN(Xtr, Xts, ytr, yts)

c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7322 - loss: 0.6433
Epoch 2/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7933 - loss: 0.5113
Epoch 3/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7933 - loss: 0.4947
Epoch 4/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7933 - loss: 0.4796
Epoch 5/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.7933 - loss: 0.4686
Epoch 6/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7934 - loss: 0.4586
Epoch 7/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7936 - loss: 0.4484
Epoch 8/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7945 - loss: 0.4369
Epoch 9/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7952 - loss: 0.4249
Epoch 10/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8075 - loss: 0.4144
Epoch 11/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8249 - loss: 0.4033
Epoch 12/16
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step

## 2. Under Sampling

In [7]:
# Separate classes in training set
df3 = df2.copy()

sample_0 = df3[df3['churn'] == 0]
sample_1 = df3[df3['churn'] == 1]

# Perform Under-sampling
under_sample_0 = sample_0.sample(len(sample_1), random_state=2)
df_under = pd.concat([under_sample_0, sample_1], axis=0)

Xtr_under = df_under.drop('churn', axis=1)
ytr_under = df_under['churn']

Xtr, Xts, ytr, yts = train_test_split(Xtr_under, ytr_under, test_size=0.2, random_state=2, stratify=ytr_under)

# Train ANN on Under-Sampled data
y_predict_under = ANN(Xtr, Xts, ytr, yts)

c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.4784 - loss: 0.6928
Epoch 2/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5219 - loss: 0.6913
Epoch 3/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5324 - loss: 0.6894
Epoch 4/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5496 - loss: 0.6858
Epoch 5/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5809 - loss: 0.6755
Epoch 6/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6128 - loss: 0.6593
Epoch 7/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6364 - loss: 0.6453
Epoch 8/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6459 - loss: 0.6333
Epoch 9/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6585 - loss: 0.6233
Epoch 10/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6619 - loss: 0.6148
Epoch 11/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6668 - loss: 0.6080
Epoch 12/16
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

## 3. Over-Sampling

In [8]:
sample_0 = df3[df3['churn'] == 0]
sample_1 = df3[df3['churn'] == 1]

# Perform Over-sampling on minority class
sample_1_over = sample_1.sample(len(sample_0), replace=True, random_state=2)
df_over = pd.concat([sample_0, sample_1_over], axis=0)

Xtr_over = df_over.drop('churn', axis=1)
ytr_over = df_over['churn']

Xtr, Xts, ytr, yts = train_test_split(Xtr_over, ytr_over, test_size=0.2, random_state=2, stratify=ytr_over)

# Train ANN on Over-Sampled data
y_predict_over = ANN(Xtr_over, Xts, ytr_over, yts)

c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.5595 - loss: 0.6879
Epoch 2/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6496 - loss: 0.6487
Epoch 3/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6826 - loss: 0.6155
Epoch 4/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7085 - loss: 0.5855
Epoch 5/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7305 - loss: 0.5597
Epoch 6/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7412 - loss: 0.5430
Epoch 7/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7505 - loss: 0.5291
Epoch 8/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7535 - loss: 0.5215
Epoch 9/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7578 - loss: 0.5135
Epoch 10/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7606 - loss: 0.5084
Epoch 11/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7621 - loss: 0.5062
Epoch 12/16
498/498 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step

In [9]:
Xtr_over.shape, ytr_over.shape, sample_1.shape, sample_0.shape, df_over.shape

((15926, 12), (15926,), (2037, 13), (7963, 13), (15926, 13))

## 4. SMOTE (Synthetic Minority Over-sampling Technique)

In [16]:
from imblearn.over_sampling import SMOTE


# Re-spliting the data
y = df3.churn
X = df3.drop('churn', axis=1)

smote = SMOTE(sampling_strategy='minority', random_state=2)
X_smote, y_smote = smote.fit_resample(X, y)

Xtr, Xts, ytr, yts = train_test_split(X_smote, y_smote, test_size=0.2, random_state=2, stratify=y_smote)

y_predict_smote = ANN(Xtr, Xts, ytr, yts)

c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.5053 - loss: 0.6931
Epoch 2/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5454 - loss: 0.6784
Epoch 3/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6870 - loss: 0.6033
Epoch 4/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7010 - loss: 0.5828
Epoch 5/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7088 - loss: 0.5725
Epoch 6/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7114 - loss: 0.5647
Epoch 7/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7184 - loss: 0.5562
Epoch 8/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7279 - loss: 0.5468
Epoch 9/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7320 - loss: 0.5405
Epoch 10/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7419 - loss: 0.5327
Epoch 11/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7443 - loss: 0.5276
Epoch 12/16
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step

## 5. Ensemble Method (Under-sampling Ensemble)

In [11]:
sample_0 = df3[df3['churn'] == 0]
sample_1 = df3[df3['churn'] == 1]

# Define the batch sampler first to avoid NameError
def get_train_batch(df_majority, df_minority, start, end):
    df_train = pd.concat([df_majority[start:end], df_minority], axis=0)
    X_train = df_train.drop('churn', axis=1)
    y_train = df_train['churn']
    return X_train, y_train

# Train 3 models on different chunks of majority class concatenated with minority class
count_minority = len(sample_1)

# Model 1
X_train1, y_train1 = get_train_batch(sample_0, sample_1, 0, count_minority)
y_pred1 = ANN(X_train1, Xts, y_train1, yts)

# Model 2
X_train2, y_train2 = get_train_batch(sample_0, sample_1, count_minority, 2*count_minority)
y_pred2 = ANN(X_train2, Xts, y_train2, yts)

# Model 3
X_train3, y_train3 = get_train_batch(sample_0, sample_1, 2*count_minority, 3*count_minority)
y_pred3 = ANN(X_train3, Xts, y_train3, yts)

c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - accuracy: 0.4985 - loss: 0.7359
Epoch 2/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5069 - loss: 0.6934
Epoch 3/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5719 - loss: 0.6825
Epoch 4/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6269 - loss: 0.6695
Epoch 5/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6433 - loss: 0.6585
Epoch 6/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6512 - loss: 0.6496
Epoch 7/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6640 - loss: 0.6405
Epoch 8/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6686 - loss: 0.6313
Epoch 9/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6762 - loss: 0.6229
Epoch 10/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6804 - loss: 0.6151
Epoch 11/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6910 - loss: 0.6052
Epoch 12/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4941 - loss: 0.6932
Epoch 2/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4946 - loss: 0.6932
Epoch 3/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5000 - loss: 0.6932
Epoch 4/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5000 - loss: 0.6932
Epoch 5/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4941 - loss: 0.6932
Epoch 6/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4951 - loss: 0.6932
Epoch 7/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4946 - loss: 0.6932
Epoch 8/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4956 - loss: 0.6932
Epoch 9/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4887 - loss: 0.6932
Epoch 10/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4907 - loss: 0.6932
Epoch 11/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4907 - loss: 0.6932
Epoch 12/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy

c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ugwu Freedom\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

Epoch 1/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5182 - loss: 0.7951
Epoch 2/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6156 - loss: 0.6779
Epoch 3/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6399 - loss: 0.6670
Epoch 4/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6510 - loss: 0.6589
Epoch 5/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6559 - loss: 0.6522
Epoch 6/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6610 - loss: 0.6461
Epoch 7/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6566 - loss: 0.6420
Epoch 8/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6667 - loss: 0.6362
Epoch 9/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6689 - loss: 0.6310
Epoch 10/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6765 - loss: 0.6262
Epoch 11/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6760 - loss: 0.6208
Epoch 12/16
128/128 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/ste

In [12]:
y_pred_final = y_pred1.copy()
for i in range(len(y_pred1)):
    n_ones = y_pred1[i] + y_pred2[i] + y_pred3[i]
    if n_ones>1:
        y_pred_final[i] = 1
    else:
        y_pred_final[i] = 0

In [15]:
cl_rep = classification_report(yts, y_pred_final)
print(cl_rep)

              precision    recall  f1-score   support

           0       0.65      0.81      0.72      1593
           1       0.75      0.57      0.65      1593

    accuracy                           0.69      3186
   macro avg       0.70      0.69      0.68      3186
weighted avg       0.70      0.69      0.68      3186

